# Tableau → Fabric: Semantic Model Generator — Play 4

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This notebook is the **semantic model generation and deployment** stage of the pipeline.
It reads the datasource and field metadata produced by Play 3, generates a TMDL-format
semantic model definition for each datasource pointing at the Delta tables written by
Play 1, and deploys each model directly to the Fabric workspace via the Fabric REST API.

**Pipeline order:** Play 2 → Play 3 → Play 4

```
Metadata_Lakehouse (Play 2 output)
  tableau_datasources  ← model names and descriptions
  tableau_fields       ← columns, data types, roles, calculated fields
  tableau_lineage      ← upstream table names
        +
h1_ultrastore (Play 3 output)
  {datasource}_{table}  ← Delta tables (DirectLake target)
        ↓
Play 4 (this notebook)
  For each datasource:
    Generate TMDL definition
    Deploy via Fabric REST API
        ↓
Fabric Workspace
  One semantic model per Tableau datasource
  DirectLake → h1_ultrastore Delta tables
```

**What each generated semantic model contains:**
- One table per upstream source table (from Play 2 lineage)
- All columns with correct data types, summarizeBy, and hidden flags
- A `_Measures` table with DAX stubs for all Tableau calculated fields
- DirectLake connection to h1_ultrastore

**What it does NOT generate (by design):**
- Relationships — these are business logic the customer configures
- DAX translations — calculated fields are stubbed with the original Tableau formula

---

**Prerequisites**
- Play 2 has been run and Metadata_Lakehouse tables are current
- Play 3 has been run and h1_ultrastore Delta tables are current
- Fabric workspace managed identity has permission to create semantic models
- Fabric admin has enabled 'Service principals can use Fabric APIs' tenant setting

**Cells in this notebook**
1. Configuration
2. Load Play 2 metadata
3. TMDL generators
4. Main loop — generate and deploy semantic models
5. Verification


## ⚠️ Start Here — Plug In Your Variables

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `WORKSPACE_ID` | Fabric workspace GUID | Fabric workspace URL |
| `DATA_LAKEHOUSE_ID` | h1_ultrastore lakehouse GUID | Fabric REST API or lakehouse settings |
| `DATA_LAKEHOUSE_NAME` | h1_ultrastore display name | e.g. `h1_ultrastore` |
| `METADATA_LAKEHOUSE` | Play 2 metadata lakehouse name | e.g. `Metadata_Lakehouse` |
| `DATASOURCE_FILTER` | Optional list of datasource names | Leave empty `[]` to process all |
| `OVERWRITE` | Whether to overwrite existing models | `True` to update, `False` to skip existing |


## Cell 1 — Configuration

Set your Fabric workspace details here. The managed identity token is obtained
automatically — no credentials needed beyond the workspace and lakehouse GUIDs.

> 🔄 **Adapting for your environment:** Update `WORKSPACE_ID`, `DATA_LAKEHOUSE_ID`,
> and `DATA_LAKEHOUSE_NAME`. Everything else is derived automatically.

In [5]:
# ── LAKEHOUSE NAMES ───────────────────────────────────────────────────────────
DATA_LAKEHOUSE_NAME = "h1_ultrastore"      # Display name of your data lakehouse
METADATA_LAKEHOUSE  = "Metadata_Lakehouse" # Display name of your metadata lakehouse

# ── FILTER CONTROLS ──────────────────────────────────────────────────────────
DATASOURCE_FILTER   = []   # e.g. ["Superstore Datasource"] — empty = process all
OVERWRITE           = True # True = update existing models, False = skip existing

# ── FABRIC REST API ──────────────────────────────────────────────────────────
FABRIC_API = "https://api.fabric.microsoft.com/v1"

import requests
import json
import base64
import uuid
import re
import time
from datetime import datetime
from pyspark.sql.types import NullType

# Get Fabric token via managed identity
token = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Get workspace ID dynamically from notebook context
WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")

# Get data lakehouse ID by display name
lakehouses_resp = requests.get(
    f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/lakehouses",
    headers=HEADERS
)
lakehouses_resp.raise_for_status()
lakehouses = lakehouses_resp.json().get("value", [])
DATA_LAKEHOUSE_ID = next(
    (l["id"] for l in lakehouses if l["displayName"] == DATA_LAKEHOUSE_NAME), None
)
if not DATA_LAKEHOUSE_ID:
    raise ValueError(f"Lakehouse '{DATA_LAKEHOUSE_NAME}' not found in workspace")

# Derive DirectLake connection URL
DIRECTLAKE_URL = f"https://onelake.dfs.fabric.microsoft.com/{WORKSPACE_ID}/{DATA_LAKEHOUSE_ID}"
EXPRESSION_SOURCE_NAME = f"DirectLake - {DATA_LAKEHOUSE_NAME}"

print("✓ Configuration loaded")
print(f"  Workspace ID:        {WORKSPACE_ID}")
print(f"  Data lakehouse:      {DATA_LAKEHOUSE_NAME} ({DATA_LAKEHOUSE_ID})")
print(f"  Metadata lakehouse:  {METADATA_LAKEHOUSE}")
print(f"  DirectLake URL:      {DIRECTLAKE_URL}")
print(f"  Datasource filter:   {DATASOURCE_FILTER or 'all'}")
print(f"  Overwrite existing:  {OVERWRITE}")
print(f"  Fabric token:        obtained ✓")

StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 7, Finished, Available, Finished, False)

✓ Configuration loaded
  Workspace ID:        a712bac8-d5ad-4773-949e-de8531569016
  Data lakehouse:      h1_ultrastore (6281b20f-5cdc-4f2c-ab2f-87c45866571b)
  Metadata lakehouse:  Metadata_Lakehouse
  DirectLake URL:      https://onelake.dfs.fabric.microsoft.com/a712bac8-d5ad-4773-949e-de8531569016/6281b20f-5cdc-4f2c-ab2f-87c45866571b
  Datasource filter:   all
  Overwrite existing:  True
  Fabric token:        obtained ✓


## Cell 2 — Load Play 3 Metadata

Reads the datasource, field, and lineage inventory from Metadata_Lakehouse.
This drives the entire model generation loop.

In [6]:
def read_metadata_table(table_name):
    """Read a Play 2 metadata table, safely dropping void columns."""
    df = spark.sql(f"SELECT * FROM {METADATA_LAKEHOUSE}.dbo.{table_name}")
    void_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NullType)]
    if void_cols:
        df = df.drop(*void_cols)
    return df.toPandas()

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

df_datasources = read_metadata_table("tableau_datasources")
df_fields      = read_metadata_table("tableau_fields")
df_lineage     = read_metadata_table("tableau_lineage")

# Apply filter
if DATASOURCE_FILTER:
    df_datasources = df_datasources[df_datasources["name"].isin(DATASOURCE_FILTER)]

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print("✓ Metadata loaded from Play 2")
print(f"  Datasources: {len(df_datasources)}")
print(f"  Fields:      {len(df_fields)}")
print(f"  Lineage:     {len(df_lineage)}")
for _, ds in df_datasources.iterrows():
    ds_id = ds["datasource_id"]
    tables = df_lineage[
        (df_lineage["datasource_id"] == ds_id) &
        (df_lineage["relationship_type"] == "upstream_table")
    ]["related_asset_name"].tolist()
    calc_count = len(df_fields[
        (df_fields["datasource_id"] == ds_id) &
        (df_fields["field_type"] == "CalculatedField")
    ])
    print(f"  • {ds['name']} → tables: {tables}, calculated fields: {calc_count}")


StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 8, Finished, Available, Finished, False)

✓ Metadata loaded from Play 3
  Datasources: 1
  Fields:      34
  Lineage:     4
  • Superstore Datasource → tables: ['Returns', 'Orders', 'People'], calculated fields: 1


## Cell 3 — TMDL Generators

Functions that generate each TMDL file part for a semantic model.
Based on the TMDL format used by Fabric semantic models (format version 4.2).

**Data type mapping:** Tableau → TMDL
- `STRING` → `string`
- `INTEGER` → `int64`
- `REAL` → `double`
- `BOOLEAN` → `boolean`
- `DATE` → `dateTime`
- `DATETIME` → `dateTime`

**summarizeBy logic:**
- `MEASURE` → `sum`
- `DIMENSION` → `none`
- Unknown/null → `none`

In [17]:
# ── DATA TYPE MAPPING ─────────────────────────────────────────────────────────
TABLEAU_TO_TMDL_TYPE = {
    "STRING":   "string",
    "INTEGER":  "int64",
    "REAL":     "double",
    "BOOLEAN":  "boolean",
    "DATE":     "dateTime",
    "DATETIME": "dateTime",
}

def slugify(s):
    """Convert a string to a safe identifier."""
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    return s.strip('_')

def make_delta_table_name(datasource_name, table_name):
    """Match the naming convention used by Play 1."""
    return f"{slugify(datasource_name)}_{slugify(table_name)}"

def clean_col(name):
    """Match Play 1's column sanitization."""
    for ch in ["(", ")", " ", ",", ";", "{", "}", "/", "\\", "\n", "\t", "="]:
        name = name.replace(ch, "_")
    return name.strip("_")

def generate_column_tmdl(field_name, data_type, role, is_hidden):
    """Generate TMDL for a single column."""
    tmdl_type = TABLEAU_TO_TMDL_TYPE.get(data_type, "string")
    summarize = "sum" if role == "MEASURE" else "none"
    format_line = ""
    if tmdl_type == "int64":
        format_line = "\n\t\tformatString: 0"
    elif tmdl_type == "dateTime":
        format_line = "\n\t\tformatString: General Date"
    tmdl_name = clean_col(field_name)
    return f"""\n\tcolumn {tmdl_name}
\t\tdataType: {tmdl_type}{format_line}
\t\tlineageTag: {uuid.uuid4()}
\t\tsourceLineageTag: {tmdl_name}
\t\tsummarizeBy: {summarize}
\t\tsourceColumn: {tmdl_name}

\t\tannotation SummarizationSetBy = Automatic
"""

def generate_measure_tmdl(field_name, formula):
    """Generate TMDL for a DAX measure stub from a Tableau calculated field."""
    safe_formula = (formula or "").replace('"', "'")
    return f"""\n\tmeasure '{field_name}' = 0
\t\t// TODO: translate from Tableau: {safe_formula}
\t\tlineageTag: {uuid.uuid4()}

\t\tannotation SummarizationSetBy = Automatic
"""

def generate_table_tmdl(table_display_name, delta_table_name, columns_tmdl, expression_source):
    """Generate TMDL for a full table including partition."""
    return f"""table {table_display_name}
\tlineageTag: {uuid.uuid4()}
\tsourceLineageTag: [dbo].[{delta_table_name}]
{columns_tmdl}
\tpartition {delta_table_name} = entity
\t\tmode: directLake
\t\tsource
\t\t\tentityName: {delta_table_name}
\t\t\tschemaName: dbo
\t\t\texpressionSource: '{expression_source}'

"""

def generate_measure_tmdl(field_name, formula):
    """Generate TMDL for a DAX measure stub from a Tableau calculated field."""
    safe_formula = (formula or "").replace('"', "'")
    safe_name = clean_col(field_name)
    return f"""\n\tmeasure '{safe_name}' = 0
\t\tlineageTag: {uuid.uuid4()}
\t\tannotation TableauFormula = "{safe_formula}"
\t\tannotation SummarizationSetBy = Automatic
"""

def generate_expressions_tmdl(expression_name, directlake_url):
    """Generate TMDL for the DirectLake data source expression."""
    return f"""expression '{expression_name}' =
\t\tlet
\t\t    Source = AzureStorage.DataLake("{directlake_url}", [HierarchicalNavigation=true])
\t\tin
\t\t    Source
\tlineageTag: {uuid.uuid4()}

\tannotation PBI_IncludeFutureArtifacts = False

"""

def generate_model_tmdl(table_names):
    """Generate TMDL for the model root file."""
    refs = "\n".join([f"ref table {t}" for t in table_names])
    return f"""model Model
\tculture: en-US
\tdefaultPowerBIDataSourceVersion: powerBI_V3
\tsourceQueryCulture: en-US
\tdataAccessOptions
\t\tlegacyRedirects
\t\treturnErrorValuesAsNull

annotation PBI_QueryOrder = ["{EXPRESSION_SOURCE_NAME}"]

annotation __PBI_TimeIntelligenceEnabled = 1

annotation PBI_ProTooling = ["DirectLakeOnOneLakeInWeb","WebModelingEdit"]

{refs}
"""

def generate_database_tmdl():
    return "database\n\tcompatibilityLevel: 1604\n"

def generate_pbism():
    return json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json",
        "version": "4.2",
        "settings": {}
    }, indent=2)

def generate_platform(display_name):
    return json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
        "metadata": {"type": "SemanticModel", "displayName": display_name},
        "config": {"version": "2.0", "logicalId": "00000000-0000-0000-0000-000000000000"}
    }, indent=2)

def encode(text):
    """Base64 encode a TMDL string for the Fabric API payload."""
    return base64.b64encode(text.encode('utf-8')).decode('utf-8')

print("✓ TMDL generators ready")
print(f"  Data types mapped: {list(TABLEAU_TO_TMDL_TYPE.keys())}")


StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 19, Finished, Available, Finished, False)

✓ TMDL generators ready
  Data types mapped: ['STRING', 'INTEGER', 'REAL', 'BOOLEAN', 'DATE', 'DATETIME']


## Cell 4 — Main Loop: Generate and Deploy Semantic Models

For each datasource:
1. Reads upstream tables from `tableau_lineage`
2. Reads fields from `tableau_fields` — columns go to their source tables, calculated fields go to `_Measures`
3. Generates TMDL definition parts
4. Deploys via Fabric REST API `createItem` endpoint
5. Polls for completion if async

> **Note:** Relationships are intentionally not generated — the customer configures
> these based on their business logic. All tables and columns are present and ready.

> **On failure:** each model is deployed independently. If one fails the loop continues.

In [18]:
def get_existing_models():
    """Get dict of existing semantic model display names → IDs in workspace."""
    resp = requests.get(
        f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels",
        headers=HEADERS
    )
    resp.raise_for_status()
    return {m["displayName"]: m["id"] for m in resp.json().get("value", [])}

def deploy_semantic_model(display_name, parts, existing_id=None):
    """
    Deploy a semantic model via Fabric REST API.
    If existing_id provided and OVERWRITE=True, updates existing model.
    Otherwise creates new.
    Polls for async completion.
    """
    definition = {"format": "TMDL", "parts": parts}

    if existing_id and OVERWRITE:
        # Update existing model
        resp = requests.post(
            f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels/{existing_id}/updateDefinition",
            headers=HEADERS,
            json={"definition": definition}
        )
    else:
        # Create new model
        resp = requests.post(
            f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items",
            headers=HEADERS,
            json={
                "displayName": display_name,
                "type": "SemanticModel",
                "definition": definition
            }
        )

    # Handle async (202)
    if resp.status_code == 202:
        operation_url = resp.headers.get("Location")
        retry_after = int(resp.headers.get("Retry-After", 20))
        time.sleep(retry_after)
        poll = requests.get(operation_url, headers=HEADERS)
        poll.raise_for_status()
        result = poll.json()
        if result.get("status") != "Succeeded":
            raise Exception(f"Async operation failed: {result}")
        return result

    resp.raise_for_status()
    return resp.json()

def build_model_parts(ds_name, ds_id):
    """
    Build all TMDL parts for a datasource.
    Returns list of {path, payload, payloadType} dicts ready for Fabric API.
    """
    # Get upstream tables from lineage
    upstream_tables = df_lineage[
        (df_lineage["datasource_id"] == ds_id) &
        (df_lineage["relationship_type"] == "upstream_table")
    ]["related_asset_name"].tolist()

    # Get all fields for this datasource
    ds_fields = df_fields[df_fields["datasource_id"] == ds_id]

    parts = []
    table_names = []

    # ── Generate one table per upstream source ──────────────────────────────
    for table_name in upstream_tables:
        delta_table = make_delta_table_name(ds_name, table_name)
        table_fields = ds_fields[
            (ds_fields["field_type"] == "ColumnField") &
            (ds_fields["source_table"] == table_name)
        ]

        columns_tmdl = ""
        for _, field in table_fields.iterrows():
            columns_tmdl += generate_column_tmdl(
                field["field_name"],
                field.get("data_type", "STRING"),
                field.get("role", "DIMENSION"),
                field.get("is_hidden", False)
            )

        table_tmdl = generate_table_tmdl(
            table_name, delta_table, columns_tmdl, EXPRESSION_SOURCE_NAME
        )
        parts.append({
            "path": f"definition/tables/{delta_table}.tmdl",
            "payload": encode(table_tmdl),
            "payloadType": "InlineBase64"
        })
        table_names.append(table_name)

    # ── Generate _Measures table from CalculatedFields ───────────────────────
    calc_fields = ds_fields[ds_fields["field_type"] == "CalculatedField"]
    measures_tmdl = ""
    for _, field in calc_fields.iterrows():
        measures_tmdl += generate_measure_tmdl(
            field["field_name"],
            field.get("formula", "")
        )
    measures_table = generate_measures_table_tmdl(measures_tmdl)
    parts.append({
        "path": "definition/tables/_Measures.tmdl",
        "payload": encode(measures_table),
        "payloadType": "InlineBase64"
    })
    table_names.append("_Measures")

    # ── Expressions (DirectLake connection) ──────────────────────────────────
    parts.append({
        "path": "definition/expressions.tmdl",
        "payload": encode(generate_expressions_tmdl(EXPRESSION_SOURCE_NAME, DIRECTLAKE_URL)),
        "payloadType": "InlineBase64"
    })

    # ── Model root ───────────────────────────────────────────────────────────
    parts.append({
        "path": "definition/model.tmdl",
        "payload": encode(generate_model_tmdl(table_names)),
        "payloadType": "InlineBase64"
    })

    # ── Database ─────────────────────────────────────────────────────────────
    parts.append({
        "path": "definition/database.tmdl",
        "payload": encode(generate_database_tmdl()),
        "payloadType": "InlineBase64"
    })

    # ── definition.pbism ─────────────────────────────────────────────────────
    parts.append({
        "path": "definition.pbism",
        "payload": encode(generate_pbism()),
        "payloadType": "InlineBase64"
    })

    # ── .platform ────────────────────────────────────────────────────────────
    parts.append({
        "path": ".platform",
        "payload": encode(generate_platform(ds_name)),
        "payloadType": "InlineBase64"
    })

    return parts, table_names

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
results = []
errors  = []

print(f"Starting semantic model generation — {datetime.utcnow().isoformat()}")
print("=" * 60)

existing_models = get_existing_models()
print(f"  Existing models in workspace: {list(existing_models.keys())}")

for _, ds_row in df_datasources.iterrows():
    ds_id   = ds_row["datasource_id"]
    ds_name = ds_row["name"]
    model_display_name = f"{ds_name} — Fabric Semantic Model"

    print(f"\n── {ds_name} ──")

    # Check if exists
    existing_id = existing_models.get(model_display_name)
    if existing_id and not OVERWRITE:
        print(f"  ⚠ Skipped — model already exists and OVERWRITE=False")
        continue

    try:
        parts, table_names = build_model_parts(ds_name, ds_id)
        print(f"  Tables: {table_names}")
        print(f"  Parts:  {len(parts)} TMDL files generated")

        result = deploy_semantic_model(model_display_name, parts, existing_id)
        action = "updated" if existing_id else "created"
        print(f"  ✓ Semantic model {action}: '{model_display_name}'")
        results.append({"datasource": ds_name, "model": model_display_name,
                        "tables": table_names, "action": action})

    except Exception as e:
        print(f"  ✗ Failed: {e}")
        errors.append({"datasource": ds_name, "error": str(e)})

print(f"\n{'=' * 60}")
print(f"✓ Complete — {datetime.utcnow().isoformat()}")
print(f"  Models deployed: {len(results)}")
print(f"  Errors:          {len(errors)}")
if errors:
    for e in errors:
        print(f"  ✗ {e['datasource']}: {e['error']}")


StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 20, Finished, Available, Finished, False)

Starting semantic model generation — 2026-05-20T05:19:00.470549
  Existing models in workspace: ['Test Model', 'Meta_data_semantic_Model', 'Test Bim', 'Superstore Datasource — Fabric Semantic Model']

── Superstore Datasource ──
  Tables: ['Returns', 'Orders', 'People', '_Measures']
  Parts:  9 TMDL files generated
  ✓ Semantic model updated: 'Superstore Datasource — Fabric Semantic Model'

✓ Complete — 2026-05-20T05:19:21.388108
  Models deployed: 1
  Errors:          0


## Cell 5 — Verification

Confirms deployed models and lists them with their tables.

In [9]:
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

# Re-fetch models from workspace
resp = requests.get(
    f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/semanticModels",
    headers=HEADERS
)
resp.raise_for_status()
all_models = resp.json().get("value", [])

if results:
    print("\n── Models deployed this run ──")
    for r in results:
        print(f"  ✓ [{r['action']}] {r['model']}")
        print(f"         Tables: {r['tables']}")

print(f"\n── All semantic models in workspace ──")
for m in all_models:
    marker = "← new" if m["displayName"] in [r["model"] for r in results] else ""
    print(f"  • {m['displayName']} {marker}")

print(f"\n  ✓ Semantic models ready in Fabric workspace")
print(f"  ✓ DirectLake → {DATA_LAKEHOUSE_NAME}")
print(f"  ✓ Add relationships in Fabric to complete the model")


StatementMeta(, 0e8068c4-3f17-4311-b3a8-b382af87ece3, 11, Finished, Available, Finished, False)

VERIFICATION

── All semantic models in workspace ──
  • Test Model 
  • Meta_data_semantic_Model 
  • Test Bim 

  ✓ Semantic models ready in Fabric workspace
  ✓ DirectLake → h1_ultrastore
  ✓ Add relationships in Fabric to complete the model
